### Some Notes 
- **Our Task**: Text Generation/Causal Language Modeling (CLM)/Auto-regressive Modeling
    - Predicts the next word given all of the previous words/tokens/input text.
        - Take a sequence that needs me completed then *outputs*
    - Predicts one token at a time \u2192 results in natural, human-like text.
    - Typically **decoder-only / autoregressive models.
    - Better for chatbots!
    - Example Models: GPT, BLOOM, PaLM
- First, what does a tokenizer return?
    - Tokenizer returns a dict of 3 items:
        - input_ids: numbers representing tokens in text - each token gets unique number from model vocabulary.
            - Basically each word is converted into a number.
            - Later on, in embedding layer (first layer) of Transformer, model converts these individual word/token/number into a vector of meaning.
            - This long embedding vector represents the meaning, context, relationships of word
            - Some dimensions are = syntax, tense, semantics, NER, position, contextual meaning (what word emans based on surrounding words)
            - This gets updated throughout the layer
        - token_type_ids indicate which sequence a token belongs to - 0 or 1 for question, answer sequence
        - attention_mask: indicates wheteher a token should be masked or not - 
                  - 1 = model should pay attention to this token
                  - 0 = padding = ignore. Adds extra inputs to make all inputs same length
- These three things are what a model uses 
- What should the input look like?
    - Dataset: you can use any plain text dataset and then tokenize the text to prepare the data.
    - Data processing for CLM... (HuggingFace youtube video)
        - Tokenize texts so it can fit into model.
        - Different texts have different lengths.
        - But Transformer models have limited context window (how many tokens it can process at once) ...
        - Thus we must chunk the input sequences into context sized pieces
            - HuggingFace tokenizer has specific parameters to handle this: return_overflowing_tokens = True, Truncation = True, max_length = ___
            - This will automatically chunk the text to be the size of the model's context window
            - *Can add stride to prevent chunks being totally disconnected --> allow some overlab between chunks ie.) stride = 50
            - *If last chunk is too small, truncate it
        - Full Tokenize function/dataset prep:
            - Tokenize, form chunks
            - Go through each chunk, if chunk matches the length --> add to input iterator/input_ids list
            - Use batches, removing existing columns
        - Case when text samples are a lot different than context window length
            - Tokenize without truncation first
            - then concatenate the tokenized samples with an end of string token or **EOS** token
        - So far we've talked about the inputs...
        - What about the **labels**?
            - We don't need extra labels for the input text/sequences --> they are the labels!
            - Labels are the inputs but shifted left one.
            - First element of input sequence is not used as a label.
            - Last element of input sequenec also not used.
        - How do you create the labels for CLMs in code?
            -  To calculate the loss on a batch...
            -  Pass input_ids as labels --> alll the shifting is handled by the modeling internally!
        - So what does the model actually need to be fed?
            - Just the input_ids, (sometimes attnetion_mask)
            - Also labels = which are just the shifted input_ids
            - Do we have to do this by hand... NO
- What is the **data collator**?
    - Objects that form batches of data by using a list of dataset elements as input. Same type as train_dataset and eval_dataset
    - Built in processing tools --> add padding, random data augmentation etc.
    - For CLM training, we use DataCollatorForLanguageModeling --> automates several preprocessing tasks
         -  Pads sequences to the longest example in the batch (if needed).
             - If batches (for instance you pass in indiviudal sentences, or paragraphs separately into model) have different length sequences, data collator padds them to be same length. Sets label = -100 where there's padding.
         -  Creates labels from input_ids (for CLM, this is just the shifted labels --> shift left one)
             -  **The shifting automatically happens in the model's loss function** Reference video --> you pass in labels twice!
         - Also makes sure that attention_mask is applied correctly - we don't want padding to be processed.   
         -  Ignore padding in loss computation (set labels = -100 for padded tokens)
         -  Creates batches for PyTorch.
         -  Do not need mlm or masked languaged modeling --> or do we???
    - Static padding vs dynamic padding
        - Static is inefficient =  pad all sentences in the dataset to the maximum length (e.g., 512 tokens), even short sequences will have a lot of unnecessary padding.
        - Dynamic is efficient = we only pad within each batch to the longest sequence in that batch.
            - ex.) batch 1 has two sentences. Longer one has 7 tokens so shorter one is padded to have 7 also.
            - ex.) batch 2 has three sentences. Longest has 10 so shorter ones are padded to have 10 also.
    - The full DataCollatorForLanguageModeling process:
        - Input is tokenized text into input_ids
        - Split tokenized text into fixed-length sequences
        - Group sequences into batches ie.) one batch has 4 sequences
            - Dynamically adding padding to sequences in each batch
            - Makes rues that padding aren't included to loss
        - Set labels = input_ids 
        - Train the model batch by batch until we finish all sequences 
- Evaluation Metrics
    -  Traditional classification metrics not used since there isn't a single right answer.
    -  Instead evaluation the distribution of the text completed by the model"
        - Cross entropy loss
        - Perplexity = the exponential of cross entropy loss
- General Transformer Notes
    - Encoder models = use only encoder of Transformer model. At each stage, attentional layers can access *all the words** in the initial sentence --> **bidirectional attention**, thus labeled as **auto-encoding models**.
        - Typically trained through masking and then reconstructing the initial sentence.
        - Tasks: understanding of full sentence --> sentence classification, NER, word classifcation, extractive question answering
        - Examples: BERT
    - Decoder models = use only decoder of Transformer model. At each stage, for a given word, the attention layers can only access the words *position before it* i n the sentence = **auto-regressive models**.
        - Training: Predicting next word in sentence.
        - Tasks: Text generation
        - Examples: GPT models
- **Future Notes**
    - Our initial dataset format of doing first half --> second half prompt/answer is perfectly suited for our question/answering dataset.
    - Could do some hybrid approach: pretrain CLM on Martha's full writing to teach style and vocab... then fine-tune  on prompt-answer pairs from her interviews and hone in conversational skills.
    - The Fine Tuning Process:
        - LORA approach: Low-Rank Adaptation --> fine-tuning LLMs efficiently and cheaply by adding small, trainable layers instead of modifying entire model.
            - Freeze the base model, then only modify some small, trainable layers
            - The question becomes... **How can we encode and control style into these small layers on top?**
        - LoRA only modifies subset of parameters --> typically the attention layers, then freezes the rest.
          - Must carefully select which parts of models to fine-tune
            - Must design prompts and datasets to reinforce the style.
            - Use existing or custom loss function that force model to stay stylistically aligned, without a tradeoff in meaning.
            - Layer names: q_proj, v_proj, k_proj, o_proj, mlp
        - Design a training dataset that really reinforces style --> reflects style, syntax, rhythm and philosophy.
        - Style-controlled prompt engineering. There are options (can potentially control style intensity with these as well)
            - Prefix based style control - prepend special token to input prompts to guide responses. Model learns to respond differently based on the prefix
            - Embedding based control - Use Custom LoRA embeddings to bias/guide responses toward Martha's tone.
            - Train a small embedding vector for Martha.
            - Use the vector to nudge responses in right direction.
        - Use Style-Specific Loss Functions
            - Research more
            - Some have been created
        - Potentially turn the style layers into an adapter that can be plug and play.

- Recap/Overview of PyTorch TrainingArguments - configures/sets up how to train the model. "The recipe"
- Common Parameters ...
        - `output_dir="./model_output"` - Directory where the trained model & checkpoints are saved.
    - `num_train_epochs=3` - Number of times the model sees the full dataset.
    - `per_device_train_batch_size=8` - Batch size per GPU/CPU during training.
    - `per_device_eval_batch_size=8` - Batch size per GPU/CPU during evaluation.
    - `learning_rate=5e-5` - Step size for adjusting model weights.
    - `weight_decay=0.01` - Regularization to prevent overfitting.
    - `evaluation_strategy="epoch"` - When to evaluate (options: `"no"`, `"steps"`, `"epoch"`).
    - `save_strategy="epoch"` - When to save the model (options: `"no"`, `"steps"`, `"epoch"`).
    - `logging_steps=500` - How often to log training progress.
    - `save_total_limit=2` - Keeps only the latest 2 model checkpoints (deletes older ones).
    - `fp16` = 16-bit floating point precision aka **mixed precision training**
        - Reduces memory usage + speeds up training  by using half-precision 16-bit numbers instead of full precisiion 32-bit numbers. Helps with GPU training.
- Advanced Parameters (for Performance Tuning and Debugging)
    - `gradient_accumulation_steps=4` - Simulates larger batch sizes by accumulating gradients over multiple steps.
    - `warmup_steps=500` - Gradually increases learning rate at the beginning of training.
    - `logging_dir="./logs"` - Where to store training logs for visualization (e.g., TensorBoard).
    - `load_best_model_at_end=True` - Automatically loads the best model after training.
    - `metric_for_best_model="loss"` - Defines the metric used to determine the best model.
    - `disable_tqdm=True` - Disables the training progress bar (useful for notebooks or logging).
    - `adam_beta1=0.9, adam_beta2=0.999` - Custom AdamW optimizer settings.
    - `lr_scheduler_type="cosine"` - Defines learning rate decay (options: `"linear"`, `"cosine"`, `"constant"`).
    - `seed=42` - Sets a random seed for reproducibility.
    - `fp16_opt_level="O1"` - Further controls mixed precision behavior.

- Recap/Overview of PyTorch Trainer class - handles and the runs the full training process... "The chef"
    - Model Training
    - Evaluation
    - Saving/loading models
    - Logging Training progress
- Common Parameters ...
    - `model=model` - The pre-trained model to fine-tune.
    - `args=training_args` - The training configuration (`TrainingArguments` object).
    - `train_dataset=train_dataset` - The dataset used for training.
    - `eval_dataset=test_dataset` - The dataset used for evaluation.
    - `data_collator=data_collator` - Handles dynamic padding for batches.
    - `tokenizer=tokenizer` - The tokenizer used for text preprocessing.
    - `compute_metrics=compute_metrics` - Custom function for calculating evaluation metrics.
- Advcaned Parameters (for Performance tuning and debugging)
    - `optimizers=(optimizer, scheduler)` - Custom optimizer (AdamW, SGD) and learning rate scheduler.
    - `callbacks=[callback]` - Custom callbacks for additional monitoring or stopping conditions.
    - `preprocess_logits_for_metrics=logits_processor` - Custom function to modify logits before computing metrics.
    - `disable_tqdm=True` - Disables progress bars (useful for logging environments).
    - `data_collator=default_data_collator` - Uses Hugging Faces default data collator.
    - `eval_accumulation_steps=10` - Stores evaluation results every 10 steps.
    - `max_length=1024` - Maximum sequence length for model inputs.
    - `gradient_checkpointing=True` - Reduces memory usage during training (useful for large models).


### Hugging Face CLM Article Notes/Explanation
- Figure out what column/field of your dataset you need. ie.) 'text': a bunch of text ...
- Preprocess:
    - Tokenize your text according to your model (extract it however necessary)
    - Mkae 


### Example 1: HuggingFace Data Processing for Causal Language Modeling

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset, DatasetDict

ds_train = load_dataset("huggingface-course/codeparrot-ds-train", split="train")
ds_valid = load_dataset("huggingface-course/codeparrot-ds-valid", split="train")

raw_datasets = DatasetDict(
    {
        "train": ds_train,
        "valid": ds_valid,
    }
)

tokenizer = AutoTokenizer.from_pretrained("huggingface-course/code-search-net-tokenizer")
model = AutoModelForCausalLM.from_pretrained("huggingface-course/codeparrot-ds")
batch = tokenizer(["import numpy as np"], return_tensors="pt")

text = "import numpy as np\n"*20
context_length = 128

In [ ]:
outputs = tokenizer(
        text,
        truncation=True,
        max_length=16,
        return_overflowing_tokens=True,
        return_length=True,
    )

print(f"Input chunk lengths: {(outputs['length'])}")

In [ ]:
def tokenize(element):
    outputs = tokenizer(
        element["content"],
        truncation=True,
        max_length=context_length,
        return_overflowing_tokens=True,
        return_length=True,
    )
    input_batch = []
    for length, input_ids in zip(outputs["length"], outputs["input_ids"]):
        if length == context_length:
            input_batch.append(input_ids)
    return {"input_ids": input_batch}


tokenized_datasets = raw_datasets.map(
    tokenize, batched=True, remove_columns=raw_datasets["train"].column_names
)

In [ ]:
output = model(input_ids=batch["input_ids"], labels=batch["input_ids"])
loss = output.loss

### Example 2: HuggingFace CLM Article 

In [ ]:
from datasets import load_dataset

eli5 = load_dataset("eli5_category", split="train[:5000]")

In [2]:
eli5 = eli5.train_test_split(test_size=0.2)

In [3]:
eli5["train"][0]

{'q_id': '7a7op7',
 'title': 'Funko Pops. Why?',
 'selftext': '',
 'category': 'Other',
 'subreddit': 'explainlikeimfive',
 'answers': {'a_id': ['dp7romo'],
  'text': ["Their uniform designs regardless of what show, game, or movie the character is based on lends to its collectible nature. They're affordable and an easy way to decorate a small personal space as a statement of personal taste while being aesthetically appealing due to their similar size and style."],
  'score': [3],
  'text_urls': [[]]},
 'title_urls': ['url'],
 'selftext_urls': ['url']}

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2")

In [5]:
eli5 = eli5.flatten()
eli5["train"][0]

{'q_id': '7a7op7',
 'title': 'Funko Pops. Why?',
 'selftext': '',
 'category': 'Other',
 'subreddit': 'explainlikeimfive',
 'answers.a_id': ['dp7romo'],
 'answers.text': ["Their uniform designs regardless of what show, game, or movie the character is based on lends to its collectible nature. They're affordable and an easy way to decorate a small personal space as a statement of personal taste while being aesthetically appealing due to their similar size and style."],
 'answers.score': [3],
 'answers.text_urls': [[]],
 'title_urls': ['url'],
 'selftext_urls': ['url']}

In [7]:
def preprocess_function(examples):
    return tokenizer([" ".join(x) for x in examples["answers.text"]])

- What is going on below?
    - map() --> applies function to every row in dataset
    - run preprocess_function on all rows to join data together into single string. Then it converts text into tokens/token ids.
    - Batched = true --> runs preprocess_function on each batch of the dataset instead of one row at a time --> speeds up processing.
    - num_proc = 4 --> uses 4 parallel processes to tokenize data faster
    - remove_columns --> deletes all non-tokenized columns from dataset

In [ ]:
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=eli5["train"].column_names,
)

- What is going on below?
    - Block_size = maximum length of each sequence --> 128 tokens per chunk
        - We want to make sure each training example is exactly this length
    - Then we concatenate all tokenized examples because ....
        - Dataset might have multiple short tokenizes sequences
        - So we join them into one long sequence of tokens
    - Then we calculate the maximum usable length to make sure we have complete sequences
        - Drop any remainder --> tokens that don't fit into a full block_size = 128
    - Then we split text into blocks of block_size
    - Then we copy input_ids to labels 

In [14]:
block_size = 128


def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [ ]:
lm_dataset = tokenized_eli5.map(group_texts, batched=True, num_proc=4)

In [ ]:
lm_dataset['train'][0]

- What is going on below?
    - The data collator only handles batching (grouping multiple examples into a batch).
    - It does NOT modify sequence lengths -- it only pads shorter sequences in a batch to match the longest one.
    - group_texts is necessary before the data collator to ensure all sequences are of the same fixed length (block_size).
        - Some blocks might get formed that are shorter than 128 --> we must pad it to match longest sequence in the batch.
- What is data collator doing?
    - Pads shorter sequences in the batch to match the longest one.
    - Creates an attention_mask (so padding tokens are ignored).
    - Ensures labels are identical to input_ids (for CLM training).
- What does it not do? (That the group_text function must do instead)
    - Does NOT change sequnce length.
    - Does NOT group text into block_size chunks.

In [ ]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [19]:
data_collator

DataCollatorForLanguageModeling(tokenizer=GPT2TokenizerFast(name_or_path='distilbert/distilgpt2', vocab_size=50257, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
), mlm=False, mlm_probability=0.15, pad_to_multiple_of=None, tf_experimental_compile=False, return_tensors='pt')

In [ ]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")

In [ ]:
training_args = TrainingArguments(
    output_dir="my_awesome_eli5_clm-model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()

In [ ]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")